<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.4 MB/s eta 0:00:00


### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

--2026-04-26 13:24:42--  http://songs_dataset.zip/
Resolving songs_dataset.zip (songs_dataset.zip)... failed: Name or service not known.
wget: unable to resolve host address ‘songs_dataset.zip’
--2026-04-26 13:24:43--  https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip [following]
--2026-04-26 13:24:43--  https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.

In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['paul-simon.txt',
 'lil-wayne.txt',
 'lin-manuel-miranda.txt',
 'beatles.txt',
 'nursery_rhymes.txt',
 'jimi-hendrix.txt',
 'cake.txt',
 'notorious-big.txt',
 'r-kelly.txt',
 'ludacris.txt',
 'eminem.txt',
 'joni-mitchell.txt',
 'britney-spears.txt',
 'dolly-parton.txt',
 'bob-dylan.txt',
 'bjork.txt',
 'dj-khaled.txt',
 'lady-gaga.txt',
 'dr-seuss.txt',
 'al-green.txt',
 'drake.txt',
 'bruno-mars.txt',
 'nickelback.txt',
 'dickinson.txt',
 'rihanna.txt',
 'bieber.txt',
 'kanye-west.txt',
 'disney.txt',
 'notorious_big.txt',
 'bruce-springsteen.txt',
 'kanye.txt',
 'janisjoplin.txt',
 'leonard-cohen.txt',
 'missy-elliott.txt',
 'Kanye_West.txt',
 'adele.txt',
 'nicki-minaj.txt',
 'johnny-cash.txt',
 'nirvana.txt',
 'alicia-keys.txt',
 'michael-jackson.txt',
 'amy-winehouse.txt',
 'blink-182.txt',
 'bob-marley.txt',
 'radiohead.txt',
 'patti-smith.txt',
 'lorde.txt',
 'Lil_Wayne.txt',
 'prince.txt']

In [74]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/bruno-mars.txt', sep='/n', header=None)
df.head()

/tmp/ipykernel_8468/1259250787.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/bruno-mars.txt', sep='/n', header=None)


,0
0,Now greetings to the world! Standing at this l...
1,"Whiskey coming through my pores,"
2,Feeling like I run this whole block.
3,Lotto tickets cheap beer
4,"That's why you can catch me here,"


In [75]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 3270


### 1 - Preprocesamiento

In [76]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [77]:
# Demos un vistazo
sentence_tokens[:2]

[['now',
  'greetings',
  'to',
  'the',
  'world',
  'standing',
  'at',
  'this',
  'liquor',
  'store'],
 ['whiskey', 'coming', 'through', 'my', 'pores']]

### 2 - Crear los vectores (word2vec)

In [78]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [79]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [80]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [81]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 3270


In [82]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 678


### 3 - Entrenar embeddings

In [93]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=25,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 88708.84375
Loss after epoch 1: 88564.234375
Loss after epoch 2: 85533.078125
Loss after epoch 3: 84871.71875
Loss after epoch 4: 83138.0
Loss after epoch 5: 82795.1875
Loss after epoch 6: 81422.1875
Loss after epoch 7: 78675.125
Loss after epoch 8: 79377.8125
Loss after epoch 9: 78397.75
Loss after epoch 10: 79025.625
Loss after epoch 11: 78388.75
Loss after epoch 12: 76847.6875
Loss after epoch 13: 76139.125
Loss after epoch 14: 74239.375
Loss after epoch 15: 75672.75
Loss after epoch 16: 74732.125
Loss after epoch 17: 73447.125
Loss after epoch 18: 73179.0
Loss after epoch 19: 74252.125
Loss after epoch 20: 73640.625
Loss after epoch 21: 73910.625
Loss after epoch 22: 73351.125
Loss after epoch 23: 72676.75
Loss after epoch 24: 73529.625


(410194, 672500)

### 4 - Ensayar

In [117]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["head"], topn=10)

[('toe', 0.7651339173316956),
 ('voices', 0.7091745138168335),
 ('veins', 0.5957582592964172),
 ('player', 0.5844157934188843),
 ('losing', 0.5787584185600281),
 ('bright', 0.5525604486465454),
 ('ooooh', 0.5507858395576477),
 ('shoot', 0.5456333160400391),
 ('cute', 0.5429806113243103),
 ('five', 0.5401924252510071)]

In [114]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["eyes"], topn=10)

[('rolling', -0.01242595724761486),
 ("we'll", -0.019990624859929085),
 ("we're", -0.028724947944283485),
 ('would', -0.042082447558641434),
 ('baby', -0.06534187495708466),
 ('going', -0.06989926099777222),
 ('nowhere', -0.07684846222400665),
 ('let', -0.07932882010936737),
 ("gon'", -0.07984084635972977),
 ('young', -0.08087412267923355)]

In [115]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["young"], topn=10)

[('wild', 0.8163009881973267),
 ('free', 0.6984975934028625),
 ('3x', 0.6572157144546509),
 ('dumb', 0.5964058041572571),
 ('pimps', 0.5885589718818665),
 ('runaway', 0.579542338848114),
 ('hurt', 0.5418376326560974),
 ('treasure', 0.5357900261878967),
 ('run', 0.5315411686897278),
 ('long', 0.5299772620201111)]

In [116]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["lady"], topn=10)

[('fine', 0.6935808658599854),
 ('sexy', 0.6371285915374756),
 ('everywhere', 0.6236459016799927),
 ('searching', 0.5770976543426514),
 ("must've", 0.5639256238937378),
 ('pretty', 0.555340051651001),
 ('sad', 0.5515201091766357),
 ('ready', 0.5492936968803406),
 ('exactly', 0.5463408827781677),
 ('amazing', 0.5407148599624634)]

In [99]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

KeyError: "Key 'diedaa' not present in vocabulary"

In [103]:
# el método `get_vector` permite obtener los vectores:
vector_lady = w2v_model.wv.get_vector("lady")
print(vector_lady)

[-6.61410540e-02  2.51963019e-01  1.40974179e-01  1.85858443e-01
 -3.61894816e-02  8.87567177e-03  3.36979069e-02  3.86061043e-01
 -9.20348316e-02  3.77909951e-02 -1.04178123e-01 -1.22334719e-01
  1.17563957e-03 -2.30818614e-01 -1.75612181e-01 -5.02950735e-02
  1.75101319e-04 -2.13508278e-01 -5.76918200e-02  2.83662323e-02
 -7.40461573e-02  1.67320803e-01  1.94392607e-01 -1.32820114e-01
  4.90986602e-03 -1.20196752e-01 -1.55447572e-01  7.54943490e-02
  2.20857188e-02 -2.83577163e-02 -1.55921867e-02  1.36512280e-01
 -3.64043340e-02 -2.43038252e-01  4.04904597e-02 -1.50052086e-01
 -3.53086069e-02 -1.35350481e-01  1.92376435e-01  2.67832637e-01
 -1.74951609e-02  9.11933631e-02  1.52135968e-01 -3.51240970e-02
 -1.10149644e-01  5.18194996e-02  2.59492546e-01 -1.50189340e-01
 -1.71724975e-01  2.10389733e-01  5.04388101e-02  1.22469410e-01
 -6.38315380e-02 -2.03937888e-01  1.17281355e-01  4.95778263e-01
  2.16321737e-01 -1.26277031e-02 -1.92670718e-01  2.57227510e-01
 -1.05401427e-01  7.09869

In [106]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_lady)

[('lady', 1.0000001192092896),
 ('fine', 0.6935808658599854),
 ('sexy', 0.6371285915374756),
 ('everywhere', 0.6236459016799927),
 ('searching', 0.5770976543426514),
 ("must've", 0.563925564289093),
 ('pretty', 0.555340051651001),
 ('sad', 0.5515201091766357),
 ('ready', 0.5492936372756958),
 ('exactly', 0.5463408827781677)]

In [107]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["lady"], topn=10)

[('fine', 0.6935808658599854),
 ('sexy', 0.6371285915374756),
 ('everywhere', 0.6236459016799927),
 ('searching', 0.5770976543426514),
 ("must've", 0.5639256238937378),
 ('pretty', 0.555340051651001),
 ('sad', 0.5515201091766357),
 ('ready', 0.5492936968803406),
 ('exactly', 0.5463408827781677),
 ('amazing', 0.5407148599624634)]

### 5 - Visualizar agrupación de vectores

In [108]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [119]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=800
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [120]:
similares = [w for w, _ in w2v_model.wv.most_similar("head", topn=10)]

for w in similares:
    print(w, w in labels[:800])

toe True
voices True
veins True
player True
losing True
bright True
ooooh True
shoot True
cute True
five True


In [122]:
# Graficar los embedings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:200])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

ValueError: All arguments should have the same length. The length of argument `text` is 200, whereas the length of  previously-processed arguments ['x', 'y', 'z'] is 678

In [ ]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

### DESARROLLO

### IMAGEN 1:

<img src="https://raw.githubusercontent.com/AlexCure/PNL-Desafio2/main/img/001.png" width="800" align="center">


**Grupo de palabras: "estuvieras", "aqui", "conmigo", "ella", "todo", "lo"**

Las palabras de este grupo están bastante relacionadas con situaciones románticas o de cercanía con otra persona. Por ejemplo, "estuvieras", "aqui" y "conmigo" suelen aparecer en frases como “si estuvieras aquí conmigo”, algo muy común en canciones.

También aparece "ella", que refuerza la idea de que se está hablando de otra persona en un contexto emocional o de relación.

Por otro lado, "todo" y "lo" no tienen un significado fuerte por sí solas, por lo que agregan algo de ruido.

Como el dataset es de un solo autor, en este caso Bruno Mars, existe la posibilidad de que este grupo esté influenciado por la repetición de frases similares dentro de una o pocas canciones (cierto overfitting).

### IMAGEN 2:

<img src="https://raw.githubusercontent.com/AlexCure/PNL-Desafio2/main/img/002.png" width="800" align="center">


**Grupo de palabras: "bed", "sleep", "stay", "late", "desire", "care", "wanna", "boy", "show", "believe", "where", "choose", "why"**

Dentro de este grupo se pueden identificar algunas relaciones claras. Por un lado, "bed", "sleep", "wana", "stay" y "late" están claramente vinculadas entre sí, ya que remiten a situaciones nocturnas o inclusive emocionales, algo frecuente en letras de canciones.

También aparece "desire" y "care", que se pueden asociar nuevamente a lo emocional o afectivo, reforzando ese mismo contexto más íntimo o romántico.

El resto de las palabras como "show", "believe", "where", "choose" y "why" no tienen una relación tan clara con el grupo principal y probablemente aparezcan por co-ocurrencias frecuentes en distintas frases.

### IMAGEN 3:

<img src="https://raw.githubusercontent.com/AlexCure/PNL-Desafio2/main/img/003.png" width="800" align="center">


**Grupo de palabras: "body", "press", "hand"**

Estas palabras tienen una relación bastante clara entre sí, ya que todas remiten a contacto físico. "Body" y "hand" son partes del cuerpo, mientras que "press" sugiere una acción de acercamiento o contacto. Describen situaciones de intimidad o cercanía física, algo común en letras de canciones.

En este caso se capturé un recuadro menor ya que se encontré un grupo bastante coherente, donde el modelo logró capturar una relación semántica clara vinculada al contacto corporal.